<a href="https://colab.research.google.com/github/DURGAPRASAD-67/-ai-mentor-portfolio/blob/main/Day9_StudyAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search

import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Gemini API key: ··········


In [2]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

/tmp/ipykernel_7246/752164138.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


ImportError: Could not import ddgs python package. Please install it with `pip install -U ddgs`.

In [3]:
pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 8.4 MB/s eta 0:00:00


In [4]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

Aug 8, 2025 ... TCS BPS Hiring - Batch of 2026. TCS BPS is bringing inspiring entry-level opportunities for Arts and Commerce graduates. As part of Business Processing ... Feb 17, 2026 ... TCS NQT Hiring Announced 2026,2025,2024. 11K views · 3 months ago ...more. Ashish Kumar. 123K. Subscribe. 363. Share. Save. Report. Comments. Mar 11, 2026 ... Tata Consultancy Services (TCS) has officially annou


In [5]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_7246/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [6]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': 'a3411d72-4245-43c4-b9a6-19cae35706f1', 'type': 'tool_call'}]

[2] ToolMessage
    Content: Tata Consultancy Services (TCS), India's leading information technology company, has already offered jobs to 25,000 fresh graduates for the financial year 2027 (FY27). How to Apply for TCS NQT 2026? All interested and eligible candidates can register online before 14 June 2026 through the official T

[3] AIMessage
    Content: [{'type': 'text', 'text': 'I couldn\'t find an exact "hiring quota" for TCS in 2026. However, it was mentioned that Tata Consultancy Services (TCS) has already offered jobs to 25,000 fresh graduates for the financial year 2027 (FY27). Additionally, registration for TCS NQT 2026 is open until June 14


In [7]:
# Pass a question that should fail the tool
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

# Watch how the agent recovers
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    

[2] ToolMessage
    Fix The specified domain either does not exist or could not be contacted error during a client PC domain join using these proven solutions. Learn why emails bounce with 'domain does not exist' or 'invalid sender domain' errors. Understand common causes like DNS misconfigurations, authentication issu

[3] AIMessage
    [{'type': 'text', 'text': 'The search results indicate that the domain "this-domain-does-not-exist-12345.example.com" likely does not exist. Therefore, I cannot tell you what it says, as the URL appears to be invalid or non-existent.', 'extras': {'signature': 'CrEEAQw51scSrM+GiFwv6hdRjGdkS3+xzk1hx15


In [9]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search

In [10]:
import os
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

Enter Gemini API Key: ··········


In [11]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """
    Search the web for latest information.
    Use for current events and recent facts.
    """

    return DuckDuckGoSearchRun().run(query)

In [12]:
print(web_search.invoke({
    "query": "Latest AI news"
}))

AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwide. See the latest updates, context and perspectives about this story.All the news from the Google I/O 2026 Developer keynote. Stay ahead of the AI revolution with breaking news, expert analysis, and exclusive insights from the world of artificial intelligence. Latest. “SMBs don’t need more AI tools, they need a simpler way to use them” – this new service combines ChatGPT, Claude, Gemini, and Grok for just $20 a month. Bullet pointsCheck out "The latest AI news we announced in April" for Google's newest tech updates.Google Cloud introduced powerful new tools and chips to help businesses build AI agents.


In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [18]:
from langchain_core.tools import tool

@tool
def weather_tool(city: str) -> str:
    """
    Get weather information for a city.
    """

    return f"The weather in {city} is sunny."

In [20]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    llm,
    tools=[
        web_search,
        weather_tool
    ]
)

print("Agent Created")

Agent Created


/tmp/ipykernel_7246/2693683977.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [21]:
print("Agent Created")

Agent Created


In [28]:
result = agent.invoke({
    "messages": [
        ("user", "What is today's weather in peddapuram?")
    ]
})

for i, m in enumerate(result["messages"]):
    print(f"\n[{i}] {type(m).__name__}")

    if hasattr(m, "content"):
        print(m.content)


[0] HumanMessage
What is today's weather in peddapuram?

[1] AIMessage


[2] ToolMessage
The weather in peddapuram is sunny.

[3] AIMessage
The weather in Peddapuram is sunny.


In [29]:
for i, m in enumerate(result["messages"]):

    print(f"\n[{i}] {type(m).__name__}")

    if hasattr(m, "content"):
        print(m.content)

    if hasattr(m, "tool_calls") and m.tool_calls:
        print("Tool Calls:")
        print(m.tool_calls)


[0] HumanMessage
What is today's weather in peddapuram?

[1] AIMessage

Tool Calls:
[{'name': 'weather_tool', 'args': {'city': 'peddapuram'}, 'id': '06d20a1e-4207-4e01-87a7-ba3c2f998bed', 'type': 'tool_call'}]

[2] ToolMessage
The weather in peddapuram is sunny.

[3] AIMessage
The weather in Peddapuram is sunny.
